In [7]:
import pandas as pd
from collections import Counter
import os

# -----------------------------
# 1) Load ranked results
# -----------------------------
in_path = "Results/All_genes.csv"
df = pd.read_csv(in_path)

# Ensure all entries are strings (gene symbols)
df = df.astype(str)

n_models = df.shape[1]
n_genes = df.shape[0]

print(f"Input: {in_path}")
print(f"Models (columns): {n_models}")
print(f"Genes per model (rows): {n_genes}")

# -----------------------------
# 2) Consensus function (top X% per model)
# -----------------------------
def consensus_top_genes(df, top_percent):
    """
    df: DataFrame where each column is a model ranking (best -> worst)
    top_percent: float (e.g., 0.05 for top 5%)
    Returns:
      out: DataFrame with gene, count, fraction_of_models
      top_k: number of genes taken from each model
    """
    top_k = max(int(n_genes * top_percent), 1)

    counter = Counter()
    for col in df.columns:
        top_genes = df[col].iloc[:top_k].values
        counter.update(top_genes)

    out = (
        pd.DataFrame(counter.items(), columns=["gene", "count"])
        .sort_values(["count", "gene"], ascending=[False, True])
        .reset_index(drop=True)
    )
    out["fraction_of_models"] = out["count"] / n_models
    return out, top_k

# -----------------------------
# 3) Pretty printing grouped by count
# -----------------------------
def print_consensus_grouped(res, min_count=1):
    """
    Prints ALL genes with count >= min_count, grouped by count (descending).
    """
    counts = sorted(res["count"].unique(), reverse=True)
    for c in counts:
        if c < min_count:
            continue
        block = res[res["count"] == c].copy()
        block = block.sort_values(["fraction_of_models", "gene"], ascending=[False, True])

        print(f"\n--- count = {c} (n={len(block)}) ---")
        print(block.to_string(index=False))

# -----------------------------
# 4) Run for multiple thresholds
# -----------------------------
thresholds = [0.05, 0.10, 0.20]
results = {}
top_k_map = {}

# Optional: save outputs
out_dir = "Results"
os.makedirs(out_dir, exist_ok=True)

for t in thresholds:
    res, k = consensus_top_genes(df, t)
    results[t] = res
    top_k_map[t] = k

    print(f"\n=== TOP {int(t*100)}% (top {k} genes per model) ===")
    # Set min_count=2 if you only want genes shared by at least 2 models
    print_consensus_grouped(res, min_count=1)

    # Save full table for Excel inspection
    out_csv = os.path.join(out_dir, f"consensus_top_{int(t*100)}pct.csv")
    res.to_csv(out_csv, index=False)
    print(f"\nSaved: {out_csv}")

# -----------------------------
# 5) "Common genes" between thresholds (based on their respective top_k)
# -----------------------------
# IMPORTANT: define "common" as genes that appear in the consensus table for each threshold
# (i.e., appear at least once in any model's top_k for that threshold).
genes_5 = set(results[0.05]["gene"])
genes_10 = set(results[0.10]["gene"])
# genes_20 = set(results[0.20]["gene"])

common_genes = genes_5 & genes_10  # add & genes_20 if you want all three

print("\n=== Common genes across consensus lists (5% and 10%) ===")
print(f"Number of common genes: {len(common_genes)}")
for g in sorted(common_genes):
    print(g)

# Optional: save common genes
common_out = os.path.join(out_dir, "common_genes_5pct_10pct.csv")
pd.DataFrame({"gene": sorted(common_genes)}).to_csv(common_out, index=False)
print(f"\nSaved: {common_out}")


Input: Results/All_genes.csv
Models (columns): 5
Genes per model (rows): 659

=== TOP 5% (top 32 genes per model) ===

--- count = 5 (n=4) ---
  gene  count  fraction_of_models
 HIF1A      5                 1.0
 HTR3A      5                 1.0
SLC6A3      5                 1.0
    TH      5                 1.0

--- count = 4 (n=10) ---
    gene  count  fraction_of_models
  CAMK2A      4                 0.8
    EGFR      4                 0.8
    GAD1      4                 0.8
   GRIA1      4                 0.8
   GRIA2      4                 0.8
   GRIN1      4                 0.8
  GRIN2A      4                 0.8
  GRIN2B      4                 0.8
HSP90AA1      4                 0.8
    PDYN      4                 0.8

--- count = 3 (n=18) ---
  gene  count  fraction_of_models
 ARRB1      3                 0.6
CACNG2      3                 0.6
CACNG7      3                 0.6
CACNG8      3                 0.6
CALML6      3                 0.6
   CCK      3                 0.6
 

In [11]:
import pandas as pd
from collections import Counter
import os

# -----------------------------
# 1) Load ranked results
# -----------------------------
in_path = "Results/All_genes2.csv"
df = pd.read_csv(in_path)

# Ensure all entries are strings (gene symbols)
df = df.astype(str)

n_models = df.shape[1]
n_genes = df.shape[0]

print(f"Input: {in_path}")
print(f"Models (columns): {n_models}")
print(f"Genes per model (rows): {n_genes}")

# -----------------------------
# 2) Consensus function (top X% per model)
# -----------------------------
def consensus_top_genes(df, top_percent):
    """
    df: DataFrame where each column is a model ranking (best -> worst)
    top_percent: float (e.g., 0.05 for top 5%)
    Returns:
      out: DataFrame with gene, count, fraction_of_models
      top_k: number of genes taken from each model
    """
    top_k = max(int(n_genes * top_percent), 1)

    counter = Counter()
    for col in df.columns:
        top_genes = df[col].iloc[:top_k].values
        counter.update(top_genes)

    out = (
        pd.DataFrame(counter.items(), columns=["gene", "count"])
        .sort_values(["count", "gene"], ascending=[False, True])
        .reset_index(drop=True)
    )
    out["fraction_of_models"] = out["count"] / n_models
    return out, top_k

# -----------------------------
# 3) Pretty printing grouped by count
# -----------------------------
def print_consensus_grouped(res, min_count=1):
    """
    Prints ALL genes with count >= min_count, grouped by count (descending).
    """
    counts = sorted(res["count"].unique(), reverse=True)
    for c in counts:
        if c < min_count:
            continue
        block = res[res["count"] == c].copy()
        block = block.sort_values(["fraction_of_models", "gene"], ascending=[False, True])

        print(f"\n--- count = {c} (n={len(block)}) ---")
        print(block.to_string(index=False))

# -----------------------------
# 4) Run for multiple thresholds
# -----------------------------
thresholds = [0.05, 0.10, 0.20]
results = {}
top_k_map = {}

# Optional: save outputs
out_dir = "Results"
os.makedirs(out_dir, exist_ok=True)

for t in thresholds:
    res, k = consensus_top_genes(df, t)
    results[t] = res
    top_k_map[t] = k

    print(f"\n=== TOP {int(t*100)}% (top {k} genes per model) ===")
    # Set min_count=2 if you only want genes shared by at least 2 models
    print_consensus_grouped(res, min_count=1)

    # Save full table for Excel inspection
    out_csv = os.path.join(out_dir, f"consensus_top_{int(t*100)}pct.csv")
    res.to_csv(out_csv, index=False)
    print(f"\nSaved: {out_csv}")

# -----------------------------
# 5) "Common genes" between thresholds (based on their respective top_k)
# -----------------------------
# IMPORTANT: define "common" as genes that appear in the consensus table for each threshold
# (i.e., appear at least once in any model's top_k for that threshold).
genes_5 = set(results[0.05]["gene"])
genes_10 = set(results[0.10]["gene"])
# genes_20 = set(results[0.20]["gene"])

common_genes = genes_5 & genes_10  # add & genes_20 if you want all three

print("\n=== Common genes across consensus lists (5% and 10%) ===")
print(f"Number of common genes: {len(common_genes)}")
for g in sorted(common_genes):
    print(g)

# Optional: save common genes
common_out = os.path.join(out_dir, "common_genes_5pct_10pct.csv")
pd.DataFrame({"gene": sorted(common_genes)}).to_csv(common_out, index=False)
print(f"\nSaved: {common_out}")


Input: Results/All_genes2.csv
Models (columns): 4
Genes per model (rows): 659

=== TOP 5% (top 32 genes per model) ===

--- count = 4 (n=14) ---
    gene  count  fraction_of_models
  CAMK2A      4                 1.0
    EGFR      4                 1.0
    GAD1      4                 1.0
   GRIA1      4                 1.0
   GRIA2      4                 1.0
   GRIN1      4                 1.0
  GRIN2A      4                 1.0
  GRIN2B      4                 1.0
   HIF1A      4                 1.0
HSP90AA1      4                 1.0
   HTR3A      4                 1.0
    PDYN      4                 1.0
  SLC6A3      4                 1.0
      TH      4                 1.0

--- count = 3 (n=15) ---
  gene  count  fraction_of_models
 ARRB1      3                0.75
CACNG2      3                0.75
CACNG7      3                0.75
CACNG8      3                0.75
CALML6      3                0.75
  DLG4      3                0.75
  GAD2      3                0.75
 GNG13      3    

In [60]:
import os
import time
import requests
import pandas as pd

API_KEY = os.getenv("DISGENET_API_KEY", "c9a473e9-961e-49c5-9c05-b7a020ebb00f").strip()
if not API_KEY:
    raise RuntimeError("Set DISGENET_API_KEY env var first.")

DISEASE = "C1269683"  # MDD
BASE_URL = "https://api.disgenet.com/api/v1/gda/summary"
OUT_CSV = "disgenet_MDD_C1269683_ACADEMIC_curated_sources_genes.csv"
OUT_CSV_PER_SOURCE = "disgenet_MDD_C1269683_per_source.csv"

HTTPheaders = {
    "Authorization": API_KEY,      # IMPORTANT: no Bearer
    "accept": "application/json",
}

ALLOWED_SOURCES = [
    "CLINGEN", "CLINPGX", "CLINVAR", "GENCC",
    "MGD_HUMAN", "ORPHANET", "PSYGENET", "RGD_HUMAN", "UNIPROT"
]

def call_api(params):
    while True:
        r = requests.get(BASE_URL, params=params, headers=HTTPheaders, verify=False, timeout=60)

        if r.status_code == 429:
            wait_s = int(r.headers.get("x-rate-limit-retry-after-seconds", "5"))
            print(f"Rate limit hit. Waiting {wait_s}s ...")
            time.sleep(wait_s)
            continue

        if not r.ok:
            # show useful error
            print("ERROR", r.status_code, "URL:", r.url)
            print("BODY:", r.text[:1500])
            r.raise_for_status()

        return r.json()

def fetch_source(source):
    all_rows = []
    page = 0

    while True:
        params = {
            "disease": DISEASE,          # <-- key change
            "source": source,            # <-- must be one of allowed curated sources
            "page_number": str(page),
        }
        resp = call_api(params)

        payload = resp.get("payload", [])
        if not payload:
            break

        all_rows.extend(payload)

        paging = resp.get("paging", {})
        total_pages = paging.get("totalPages", None)
        if total_pages is not None and page >= int(total_pages) - 1:
            break

        page += 1

    return all_rows

# -----------------------------
# Fetch across sources
# -----------------------------
all_rows = []
for src in ALLOWED_SOURCES:
    print(f"Fetching source={src} ...")
    rows = fetch_source(src)
    print(f"  rows: {len(rows)}")
    for r in rows:
        r["source_used"] = src
    all_rows.extend(rows)

df = pd.DataFrame(all_rows)
if df.empty:
    raise RuntimeError("No results returned across all allowed sources.")

print("Returned columns:", list(df.columns))

# -----------------------------
# Identify gene + score columns
# -----------------------------
gene_col = next((c for c in ["gene_symbol", "geneSymbol", "gene", "symbol"] if c in df.columns), None)
score_col = next((c for c in ["score", "gda_score", "gdaScore", "disgenet_score", "disgenetScore"] if c in df.columns), None)

if gene_col is None or score_col is None:
    # save raw table for inspection
    df.to_csv(OUT_CSV_PER_SOURCE, index=False)
    raise RuntimeError(f"Could not auto-detect gene/score columns; saved raw per-source table to {OUT_CSV_PER_SOURCE}")

df["gene"] = df[gene_col].astype(str)
df["score"] = pd.to_numeric(df[score_col], errors="coerce")
df = df.dropna(subset=["gene", "score"])

# Save per-source (useful for debugging/traceability)
per_source = df[["gene", "score", "source_used"]].copy()
per_source.to_csv(OUT_CSV_PER_SOURCE, index=False)

# Create a combined gene list (max score across curated sources)
combined = (
    per_source
    .groupby("gene", as_index=False)
    .agg(score=("score", "max"),
         sources=("source_used", lambda s: ",".join(sorted(set(s)))))
    .sort_values("score", ascending=False)
)

combined.to_csv(OUT_CSV, index=False)

print(f"\nSaved combined gene list: {OUT_CSV} (genes={len(combined)})")
print(f"Saved per-source detail:  {OUT_CSV_PER_SOURCE} (rows={len(per_source)})")
print("\nTop 15:")
print(combined.head(15).to_string(index=False))


Fetching source=CLINGEN ...


/opt/anaconda3/lib/python3.12/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.disgenet.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  rows: 0
Fetching source=CLINPGX ...


/opt/anaconda3/lib/python3.12/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.disgenet.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  rows: 0
Fetching source=CLINVAR ...


...

/opt/anaconda3/lib/python3.12/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.disgenet.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  rows: 0
Fetching source=UNIPROT ...


/opt/anaconda3/lib/python3.12/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.disgenet.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


  rows: 0


RuntimeError: No results returned across all allowed sources.

In [50]:
import requests, os
API_KEY= 'c9a473e9-961e-49c5-9c05-b7a020ebb00f'
url="https://api.disgenet.com/api/v1/gda/disease/C1269683"
r=requests.get(url, headers={"Accept":"application/json","Authorization":f"Bearer {API_KEY}"}, allow_redirects=False)
print(r.status_code, r.headers.get("Content-Type"), r.text[:200])


404 application/json {"message":"Resource not found"}


In [68]:
from scipy.stats import hypergeom

N = 1020
K = 165
n = 32
k = 14

p_value = hypergeom.sf(k-1, N, K, n)

print(p_value)

0.0001619532539735491


In [74]:
import networkx as nx
import matplotlib.pyplot as plt

# Top consensus genes (count >= 4)
top_genes = [
    "EGFR","HTR3A","TH",
    "CACNG2","CACNG3","CACNG7","CACNG8","CALML6","CAMK2A","CCK","CCND1",
    "DLG2","DLG3","DLG4","GAD1","GAD2","GRIA1","GRIA2","GRIA4","GRIK2","GRIK4",
    "GRIN1","GRIN2A","GRIN2B","GRIN2C","GRIN2D","GRM1","HIF1A","HSP90AA1","MTOR",
    "OPRM1","SLC6A3"
]

# Keep only genes that exist in the graph
top_genes_in_graph = [g for g in top_genes if g in G.nodes()]

# Build induced subgraph
subG = G.subgraph(top_genes_in_graph).copy()

# Optional: keep only the largest connected component for a cleaner figure
if len(subG) > 0 and not nx.is_empty(subG):
    largest_cc = max(nx.connected_components(subG), key=len)
    subG = subG.subgraph(largest_cc).copy()

# Assign rough functional groups
def gene_group(g):
    if g.startswith("GRIN") or g.startswith("GRIA") or g.startswith("GRIK") or g == "GRM1":
        return "Glutamate receptors"
    elif g.startswith("DLG") or g.startswith("CACNG") or g in {"CAMK2A", "CALML6"}:
        return "Synaptic scaffolding/signaling"
    elif g in {"GAD1", "GAD2", "HTR3A", "TH", "OPRM1", "SLC6A3", "CCK"}:
        return "Neurotransmission/modulation"
    else:
        return "Other signaling"

group_to_nodes = {}
for g in subG.nodes():
    grp = gene_group(g)
    group_to_nodes.setdefault(grp, []).append(g)

# Use default matplotlib colors by category order
groups = list(group_to_nodes.keys())

pos = nx.spring_layout(subG, seed=42, k=0.6)

plt.figure(figsize=(11, 8))

for grp in groups:
    nx.draw_networkx_nodes(
        subG,
        pos,
        nodelist=group_to_nodes[grp],
        label=grp,
        node_size=700,
        alpha=0.9
    )

nx.draw_networkx_edges(subG, pos, alpha=0.4, width=1.2)
nx.draw_networkx_labels(subG, pos, font_size=9)

plt.title("Subnetwork of highly prioritized genes")
plt.legend(scatterpoints=1, frameon=True)
plt.axis("off")
plt.tight_layout()
plt.show()

NameError: name 'G' is not defined